In [1]:
import numpy as np

def write_mobius_obj(
    filename="data/surfaces/mobius-procedural.obj",
    R=1.0,
    width=0.25,
    n_theta=56,
    n_width=3
):
    vertices = []
    faces = []

    thetas = np.linspace(0, 2*np.pi, n_theta, endpoint=False)
    ws = np.linspace(-width, width, n_width)

    # Build vertices
    for i, theta in enumerate(thetas):
        c = np.array([
            R * np.cos(theta),
            R * np.sin(theta),
            0.0
        ])

        n0 = np.array([ np.cos(theta), np.sin(theta), 0])
        b0 = np.array([0.0, 0.0, 1.0])

        phi = theta / 2.0
        n = np.cos(phi) * n0 + np.sin(phi) * b0

        for w in ws:
            p = c + w * n
            vertices.append(tuple(p))

    def vid(i, j):
        return i * n_width + j + 1  # OBJ is 1-based

    # Möbius strip faces
    for i in range(n_theta):
        i_next = (i + 1) % n_theta

        for j in range(n_width - 1):
            if i_next == 0:
                jn0 = n_width - 1 - j
                jn1 = n_width - 2 - j
            else:
                jn0 = j
                jn1 = j + 1

            v00 = vid(i, j)
            v01 = vid(i, j + 1)
            v10 = vid(i_next, jn0)
            v11 = vid(i_next, jn1)

            faces.append((v00, v10, v11))
            faces.append((v00, v11, v01))

    if False:

        # -------------------------------------------------
        # Add origin vertex
        # -------------------------------------------------
        origin_index = len(vertices) + 1
        vertices.append((0.0, 0.0, 4.0))
    
        # -------------------------------------------------
        # Build true Möbius boundary loop
        # -------------------------------------------------
        boundary = []
    
        # along j = 0
        for i in range(n_theta):
            boundary.append(vid(i, 0))
    
        # seam jump
        boundary.append(vid(0, n_width - 1))
    
        # along j = 1
        for i in range(1, n_theta):
            boundary.append(vid(i, n_width - 1))
    
        # -------------------------------------------------
        # Fan triangles from boundary edges to origin
        # -------------------------------------------------
        for k in range(len(boundary)):
            v0 = boundary[k]
            v1 = boundary[(k + 1) % len(boundary)]
            faces.append((v0, v1, origin_index))

    # Write OBJ
    with open(filename, "w") as f:
        f.write("# Möbius strip capped to origin\n")
        for v in vertices:
            f.write(f"v {v[0]} {v[1]} {v[2]}\n")
        for a, b, c in faces:
            f.write(f"f {a} {b} {c}\n")

    print(f"Wrote {filename}: {len(vertices)} vertices, {len(faces)} triangles")


if __name__ == "__main__":
    write_mobius_obj()


Wrote data/surfaces/mobius-procedural.obj: 168 vertices, 224 triangles
